In [ ]:
# Enrich Bitcoin TXIDs with On-Chain Timestamps

This notebook loads a spreadsheet (CSV/XLSX) with a `txid` column, fetches on-chain timestamps from BigQuery's public
Bitcoin dataset, and writes an enriched output. It also includes optional cleanup steps for `from`, `to`, and `amount`.


In [12]:
# Optional installs (code, run only if needed)
# If running in a fresh environment, uncomment:
!pip install -r ../requirements.txt

In [13]:
# Imports, paths, and config (code)
from pathlib import Path
import os
import sys
from typing import Optional, List

import pandas as pd
from dotenv import load_dotenv

# Project paths (this notebook lives in .../btc_txid_tools/notebooks)
BASE = Path.cwd().parent
INPUT_DIR = BASE / "data" / "input"
OUTPUT_DIR = BASE / "data" / "output"

# Access helper module
sys.path.append(str((BASE / "src").resolve()))
from bq_utils import get_bigquery_client, fetch_tx_timestamps

# Config
load_dotenv()
BQ_LOCATION = os.getenv("BQ_LOCATION", "US")
BQ_PROJECT_ID: Optional[str] = os.getenv("BQ_PROJECT_ID")
BATCH_SIZE = 5000
TIMESTAMP_COL = "timestamp"
DISABLE_BQ = False  # set True to skip BigQuery (timestamps blank)

INPUT_DIR, OUTPUT_DIR

(WindowsPath('C:/Users/cisco/Desktop/btc_txid_tools/data/input'),
 WindowsPath('C:/Users/cisco/Desktop/btc_txid_tools/data/output'))

In [14]:
#Select input and sheet (code)
# Update these if needed
input_path = INPUT_DIR / "transactions.csv"  # or .xlsx
sheet_name = 0  # if Excel: use index or string sheet name

print("Input path:", input_path)
assert input_path.exists(), f"Not found: {input_path}"

Input path: C:\Users\cisco\Desktop\btc_txid_tools\data\input\transactions.csv


In [15]:

# Supports CSV/XLSX
if input_path.suffix.lower() == ".csv":
    df = pd.read_csv(input_path)
elif input_path.suffix.lower() in (".xlsx", ".xls"):
    df = pd.read_excel(input_path, sheet_name=sheet_name)
else:
    raise ValueError(f"Unsupported file type: {input_path.suffix}")

# Require 'txid' column
if "txid" not in df.columns:
    raise KeyError("Input must contain a 'txid' column")

print("Rows:", len(df))
df.head(10)

Rows: 2072


,txid,from,to,amount
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...,3HLqRkgagCCeChBBmWuDkp54YR1LEz7NuK,17AuTmGhZ4HBoJtntpN9n6vHkP5LNnrt5q,5.850062
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...,3JEirUZxifRHf7FCY2Z79e3k8FMgGM9ehY,1L14Rw9G4P173tTZdMKsFPpMNY9WdiLKK7,7.482093
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...,36tGwveXwuEnmagxrraUv2XKJXeAEzP9iB,1G1yQeKMY8mH1H5v1EbmaSegpBML8T8LuR,7.586615
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...,3F14ooo4Z4BchZrFNSxF5jgssMjJcKPfay,1PsMd9HzNxfyFNRQRxsjjvvzX48EmZVn9n,13.204210
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...,3A5zJnUeZqqjrNJCAZZkKYxh6ZgdcAKps3,1Pvw3xFhAS2i58fSdSJ7ZuHXHLSe62KcKx,5.716555
5,00d4189eca25abcf1381bff4db78db8a8edf8f11c5deb0...,35wqRthgict67zRChT3euuL3Z59uMKSryv,1NCzDib1QNF8dRD2WErgV4HSZkgyci5J8Y,59.997146
6,00e43489afb85a0d3bd05f3a30b2280416d7d333d52840...,3CejBDbGoEPsmyjmnv1YuML4prFg6hi2rg,1P4jiBBQ4X3VAmueSv673cxEimxFfDFYMK,13.933099
7,00fa784235855c1923e85557be5c5b5909eadcbd199352...,39pj2d5ZuzkCXV23BaG9Re6X8xD4vmhFNc,16T27MoP4tpPeK4C9dtDCZPqGdrG2YYmMP,7.499521
8,0113e56efa662e7afb75ab00d3494cbd573e6334db8b61...,38i9VmL9F9phFieV9HKZcBSZsW2LnVWJ49,1rJmrG2Ys22pELEHzRftdfaruxuTqU2qm,87.101273
9,011c0366033ac9d0a5ca5a878a3f16f65495942b2a50d1...,3PuTHoR6WphgBYJAwd1XfC5uc3QnpJxMWf,1yUW75KTCyDvDLUG4fqYiZ9SoQppi1UUi,32.079762


In [16]:

# Load dataset (code)
# Supports CSV/XLSX
if input_path.suffix.lower() == ".csv":
    df = pd.read_csv(input_path)
elif input_path.suffix.lower() in (".xlsx", ".xls"):
    df = pd.read_excel(input_path, sheet_name=sheet_name)
else:
    raise ValueError(f"Unsupported file type: {input_path.suffix}")

# Require 'txid' column
if "txid" not in df.columns:
    raise KeyError("Input must contain a 'txid' column")

print("Rows:", len(df))
df.head(10)


Rows: 2072


,txid,from,to,amount
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...,3HLqRkgagCCeChBBmWuDkp54YR1LEz7NuK,17AuTmGhZ4HBoJtntpN9n6vHkP5LNnrt5q,5.850062
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...,3JEirUZxifRHf7FCY2Z79e3k8FMgGM9ehY,1L14Rw9G4P173tTZdMKsFPpMNY9WdiLKK7,7.482093
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...,36tGwveXwuEnmagxrraUv2XKJXeAEzP9iB,1G1yQeKMY8mH1H5v1EbmaSegpBML8T8LuR,7.586615
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...,3F14ooo4Z4BchZrFNSxF5jgssMjJcKPfay,1PsMd9HzNxfyFNRQRxsjjvvzX48EmZVn9n,13.204210
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...,3A5zJnUeZqqjrNJCAZZkKYxh6ZgdcAKps3,1Pvw3xFhAS2i58fSdSJ7ZuHXHLSe62KcKx,5.716555
5,00d4189eca25abcf1381bff4db78db8a8edf8f11c5deb0...,35wqRthgict67zRChT3euuL3Z59uMKSryv,1NCzDib1QNF8dRD2WErgV4HSZkgyci5J8Y,59.997146
6,00e43489afb85a0d3bd05f3a30b2280416d7d333d52840...,3CejBDbGoEPsmyjmnv1YuML4prFg6hi2rg,1P4jiBBQ4X3VAmueSv673cxEimxFfDFYMK,13.933099
7,00fa784235855c1923e85557be5c5b5909eadcbd199352...,39pj2d5ZuzkCXV23BaG9Re6X8xD4vmhFNc,16T27MoP4tpPeK4C9dtDCZPqGdrG2YYmMP,7.499521
8,0113e56efa662e7afb75ab00d3494cbd573e6334db8b61...,38i9VmL9F9phFieV9HKZcBSZsW2LnVWJ49,1rJmrG2Ys22pELEHzRftdfaruxuTqU2qm,87.101273
9,011c0366033ac9d0a5ca5a878a3f16f65495942b2a50d1...,3PuTHoR6WphgBYJAwd1XfC5uc3QnpJxMWf,1yUW75KTCyDvDLUG4fqYiZ9SoQppi1UUi,32.079762


In [17]:
##############Optional cleanup/normalization of other columns (code)
# Examples: trim whitespace and normalize case on 'from'/'to'; coerce 'amount' numeric
for col in ["from", "to"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

if "amount" in df.columns:
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Optional: drop duplicate rows (exact duplicates)
# df = df.drop_duplicates().reset_index(drop=True)

# Optional: reorder columns (put txid first)
cols = list(df.columns)
if "txid" in cols:
    cols = ["txid"] + [c for c in cols if c != "txid"]
    df = df[cols]

print("After optional cleanup:", df.shape)
df.head(5)


After optional cleanup: (2072, 4)


,txid,from,to,amount
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...,3HLqRkgagCCeChBBmWuDkp54YR1LEz7NuK,17AuTmGhZ4HBoJtntpN9n6vHkP5LNnrt5q,5.850062
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...,3JEirUZxifRHf7FCY2Z79e3k8FMgGM9ehY,1L14Rw9G4P173tTZdMKsFPpMNY9WdiLKK7,7.482093
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...,36tGwveXwuEnmagxrraUv2XKJXeAEzP9iB,1G1yQeKMY8mH1H5v1EbmaSegpBML8T8LuR,7.586615
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...,3F14ooo4Z4BchZrFNSxF5jgssMjJcKPfay,1PsMd9HzNxfyFNRQRxsjjvvzX48EmZVn9n,13.204210
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...,3A5zJnUeZqqjrNJCAZZkKYxh6ZgdcAKps3,1Pvw3xFhAS2i58fSdSJ7ZuHXHLSe62KcKx,5.716555


In [18]:
# Normalize txids and prep list (code)
def normalize_txid(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower().startswith("0x"):
        s = s[2:]
    return s.upper()

df["txid_norm"] = df["txid"].apply(normalize_txid)
unique_txids: List[str] = sorted(set(t for t in df["txid_norm"].tolist() if t))

print("Unique normalized txids:", len(unique_txids))
df[["txid", "txid_norm"]].head(10)


Unique normalized txids: 2072


,txid,txid_norm
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...,002740A49E16FC127B0A58A887E5AD77CC4D5114A38AA6...
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...,002EAECEBF435E9BC3E2F3C4804EE6DD89C1DC342585D2...
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...,0038CD4166D80A95D6BF1420BEB9D16C9FB5CEC19FA31F...
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...,0083ABB19CA6CE1FF1DD55FFA6531FE4D88D3EC884D93E...
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...,00934630D88A064F60A9D294DE072D5FE49620F2D6D727...
5,00d4189eca25abcf1381bff4db78db8a8edf8f11c5deb0...,00D4189ECA25ABCF1381BFF4DB78DB8A8EDF8F11C5DEB0...
6,00e43489afb85a0d3bd05f3a30b2280416d7d333d52840...,00E43489AFB85A0D3BD05F3A30B2280416D7D333D52840...
7,00fa784235855c1923e85557be5c5b5909eadcbd199352...,00FA784235855C1923E85557BE5C5B5909EADCBD199352...
8,0113e56efa662e7afb75ab00d3494cbd573e6334db8b61...,0113E56EFA662E7AFB75AB00D3494CBD573E6334DB8B61...
9,011c0366033ac9d0a5ca5a878a3f16f65495942b2a50d1...,011C0366033AC9D0A5CA5A878A3F16F65495942B2A50D1...


In [19]:
# Auth already completed via: gcloud auth application-default login

In [20]:
## Initialize BigQuery client (code) run 'gcloud auth application-default login' in conda 
BQ_LOCATION = "US"  # required for bigquery-public-data (US)
client = None
if not DISABLE_BQ and unique_txids:
    try:
        client = get_bigquery_client(BQ_PROJECT_ID)
        _ = client.query("SELECT 1").result()  # sanity check
        print("BigQuery client ready; project:", getattr(client, "project", BQ_PROJECT_ID))
    except Exception as e:
        print("BigQuery not available:", e)
        client = None
else:
    print("BigQuery disabled or no txids; timestamps will be blank.")


BigQuery client ready; project: bitfinex-hack-tracing


In [21]:
# Skipped: hash column probe not needed; using TO_HEX(hash) via fetch_tx_timestamps().

In [22]:
# Skipped: unused hash/bytes helpers removed in favor of fetch_tx_timestamps().

In [31]:

# Removed the 'location' parameter since fetch_tx_timestamps() doesn't accept it
txid_to_ts = fetch_tx_timestamps(client, unique_txids, batch_size=BATCH_SIZE)
print("Fetched:", len(txid_to_ts), "of", len(unique_txids))


BadRequest: 400 Syntax error: Expected ")" but got keyword HASH at [2:23]; reason: invalidQuery, location: query, message: Syntax error: Expected ")" but got keyword HASH at [2:23]

Location: US
Job ID: e9790aa7-55b8-480b-96d6-5fb63752b2ff


In [24]:
# Merge timestamps and optional datetime column (code)
out = df.copy()
out[TIMESTAMP_COL] = out["txid_norm"].map(txid_to_ts).fillna("")
# Optional: a parsed datetime column for convenience
try:
    out["timestamp_dt"] = pd.to_datetime(out[TIMESTAMP_COL], utc=True, errors="coerce")
except Exception:
    pass

out = out.drop(columns=["txid_norm"])
out.head(10)

NameError: name 'txid_to_ts' is not defined

In [ ]:
Cell 11 — Write outputs (CSV and XLSX) (code)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "transactions_with_timestamps.csv"
xlsx_path = OUTPUT_DIR / "transactions_with_timestamps.xlsx"

out.to_csv(csv_path, index=False)
try:
    out.to_excel(xlsx_path, index=False)
except Exception as e:
    print("Excel write skipped:", e)

csv_path, xlsx_path

In [ ]:
Cell 12 — Quick validation and next steps (code)
total = len(out)
missing = int(out[TIMESTAMP_COL].eq("").sum())
print(f"Rows: {total}  |  Missing timestamps: {missing}  |  Coverage: {100*(total-missing)/max(1,total):.1f}%")
out.sample(min(5, total), random_state=42)